# SVM Classifier

Import library and settings

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    roc_curve
)

# Snygg stil på grafer
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

Load data and create timezones variables (Calenderdata)

In [ ]:
# Läs in datasetet
df = pd.read_csv("../dataset/all_zones_complete_2025.csv")

# Konvertera timestamp till datetime (använd timestamp_local för korrekta timmar per dygn)
df['timestamp_local'] = pd.to_datetime(df['timestamp_local'])

# Extrahera kalenderdata
df['hour'] = df['timestamp_local'].dt.hour
df['dayofweek'] = df['timestamp_local'].dt.dayofweek
df['month'] = df['timestamp_local'].dt.month
df['date'] = df['timestamp_local'].dt.date

print(f"Dataset laddat. Antal rader: {len(df)}")
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '.dataset/all_zones_complete_2025.csv'

Create goal variabel (Target) - `optimal_timme` for SE3 (or choosen zone)
We focus on prehaps electric-price-zon 3 (`price_se3_eur_mwh`) and define the 6 cheapest hours on a day like optimal (1).

In [ ]:
TARGET_ZONE = 'price_se3_eur_mwh'
HOURS_TO_SELECT = 6  # T.ex. de 6 billigaste timmarna per dygn

# Skapa kolumn för datum om den saknas
df['date'] = pd.to_datetime(df['timestamp_local']).dt.date

# Beräkna tröskel / rank per dygn för att hitta de billigaste timmarna
df['price_rank'] = df.groupby('date')[TARGET_ZONE].rank(method='min', ascending=True)
df['optimal_timme'] = (df['price_rank'] <= HOURS_TO_SELECT).astype(int)

print("Fördelning av målvariabeln (0 = ej optimal, 1 = optimal):")
print(df['optimal_timme'].value_counts())

Feature Engineering and prestart of data

In [ ]:
# Välj ut relevanta features (väder för SE3 + kalenderdata)
feature_cols = [
    'temp_se3_c', 'wind_se3_kmh', 'rain_se3_mm', 
    'hour', 'dayofweek', 'month'
]

X = df[feature_cols]
y = df['optimal_timme']

# Kronologisk eller slumpad uppdelning? 
# För tidsserier är det ofta bäst med kronologisk split, men här kör vi train/val/test (t.ex. 70% train, 15% val, 15% test)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42, shuffle=True, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1764, random_state=42, shuffle=True, stratify=y_train_val) 
# 0.1764 av 85% blir ca 15% av totalen för validering

print(f"Träningsset: {X_train.shape[0]} rader")
print(f"Valideringsset: {X_val.shape[0]} rader")
print(f"Testset: {X_test.shape[0]} rader")

Scaling of data (Important for SVM)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Hyperparam optimization with GridSearchCV (on Train/Val)
GridSearchCV search for the best `C`and `gamma` for one RBF-kernal in SVM.

In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf']
}

# Vi använder modell med klassvikt om klasserna är obalanserade
svm = SVC(class_weight='balanced', probability=True, random_state=42)

grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("\nBästa parametrar hittade:")
print(grid_search.best_params_)

best_model = grid_search.best_estimator_

Evaluation on Val- and Testset
Because it´s classification we use (Accuracy, Precision, Recall, F1 and ROC-AUC) instead of RMSE/MSE.

In [ ]:
def evaluate_model(model, X_data, y_data, dataset_name="Test"):
    y_pred = model.predict(X_data)
    y_prob = model.predict_proba(X_data)[:, 1]
    
    print(f"--- Utvärdering på {dataset_name} ---")
    print(f"Accuracy:  {accuracy_score(y_data, y_pred):.4f}")
    print(f"Precision: {precision_score(y_data, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_data, y_pred):.4f}")
    print(f"F1-score:  {f1_score(y_data, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_data, y_prob):.4f}\n")
    
    print("Classification Report:")
    print(classification_report(y_data, y_pred))
    
    return y_pred, y_prob

# Utvärdera på valideringsset
_ = evaluate_model(best_model, X_val_scaled, y_val, dataset_name="Valideringsset")

# Utvärdera slutgiltigt på testset
y_pred_test, y_prob_test = evaluate_model(best_model, X_test_scaled, y_test, dataset_name="Testset")

Visualization (Confusion Matrix and ROC-curv)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Confusion Matrix (Testset)')
axes[0].set_xlabel('Predikterad klass')
axes[0].set_ylabel('Faktisk klass')

# 2. ROC-kurva
fpr, tpr, thresholds = roc_curve(y_test, y_prob_test)
auc_val = roc_auc_score(y_test, y_prob_test)
axes[1].plot(fpr, tpr, label=f'SVM (AUC = {auc_val:.3f})', color='darkorange', lw=2)
axes[1].plot([0, 1], [0, 1], 'k--', lw=2)
axes[1].set_title('ROC-kurva')
axes[1].set_xlabel('Falskt positiv rate (FPR)')
axes[1].set_ylabel('Sant positiv rate (TPR)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()